In [ ]:
import sys
import os

# Add the 'src' directory to the Python path
sys.path.append(os.path.abspath("../src"))

### Configurations
TRAIN = False
PROVIDER = "openai"  # "openai" or "anthropic" or "gemini"
EMBEDDING_PATH = f"./SavedEmbeddings/{PROVIDER}_news.pkl"

In [2]:
# select your own API key here. (Note: This specific code will not work for you unless you specified an environment variable for OPENAI_API_KEY)
import os
from dotenv import load_dotenv

load_dotenv()

if PROVIDER == "openai":
    api_key = os.environ.get('OPENAI_API_KEY')
    prompting_model = "gpt-3.5-turbo-16k"
    embedding_model = "text-embedding-ada-002"
elif PROVIDER == "gemini":
    api_key = os.environ.get('GEMINI_API_KEY')
    prompting_model = "gemini-2.0-flash-lite"
    embedding_model = "gemini-embedding-001"
elif PROVIDER == "anthropic":
    api_key = os.environ.get('ANTHROPIC_API_KEY')
    prompting_model = "claude-sonnet-4-20250514"
    embedding_model = "all-MiniLM-L6-v2"

In [3]:
from sklearn.datasets import fetch_20newsgroups

data = fetch_20newsgroups(subset='all', remove=('headers', 'footers', 'quotes'))
corpus = data['data']
topic_names = data.target_names  # e.g., ['alt.atheism', 'comp.graphics', ...]

filtered_corpus = []
filtered_labels = []

for doc, label in zip(corpus, data.target):
    if doc != "":
        filtered_corpus.append(doc)
        filtered_labels.append(label)
        

# Now, filtered_corpus contains only non-empty docs,
# and filtered_labels contains their corresponding ground-truth labels.
corpus = filtered_corpus
labels = filtered_labels

In [4]:
import pandas as pd

# Create a dataframe with corpus and labels for viewing purposes
df = pd.DataFrame({
    'topic_name': [topic_names[label] for label in labels],
    'document': corpus,
    'label': labels
})

print(f"DataFrame shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print("\nFirst few rows:")
df

DataFrame shape: (18466, 3)
Columns: ['topic_name', 'document', 'label']

First few rows:


,topic_name,document,label
0,rec.sport.hockey,\n\nI am sure some bashers of Pens fans are pr...,10
1,comp.sys.ibm.pc.hardware,My brother is in the market for a high-perform...,3
2,talk.politics.mideast,\n\n\n\n\tFinally you said what you dream abou...,17
3,comp.sys.ibm.pc.hardware,\nThink!\n\nIt's the SCSI card doing the DMA t...,3
4,comp.sys.mac.hardware,1) I have an old Jasmine drive which I cann...,4
...,...,...,...
18461,sci.med,DN> From: nyeda@cnsvax.uwec.edu (David Nye)\nD...,13
18462,sci.electronics,\nNot in isolated ground recepticles (usually ...,12
18463,comp.sys.ibm.pc.hardware,I just installed a DX2-66 CPU in a clone mothe...,3
18464,comp.graphics,\nWouldn't this require a hyper-sphere. In 3-...,1


## Using Traditional Methods: LDA, K-means. 

TF-IDF features/matric -> K-Means/LDA -> Top-words

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
import numpy as np
from collections import Counter

import matplotlib.pyplot as plt

# Prepare the data for traditional topic modeling
print("Preparing data for topic modeling...")
print(f"Number of documents: {len(corpus)}")
print(f"Number of true topics: {len(topic_names)}")

# Create TF-IDF features
print("\nCreating TF-IDF features...")
vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words='english',
    min_df=2,
    max_df=0.95,
    ngram_range=(1, 2)
)
tfidf_matrix = vectorizer.fit_transform(corpus)
feature_names = vectorizer.get_feature_names_out()
print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")

# K-Means Clustering
print("\nPerforming K-Means clustering...")
kmeans = KMeans(n_clusters=20, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(tfidf_matrix)

# LDA Topic Modeling
print("Performing LDA topic modeling...")
lda = LatentDirichletAllocation(
    n_components=20,
    random_state=42,
    max_iter=100,
    learning_method='batch'
)
lda.fit(tfidf_matrix)
lda_labels = lda.transform(tfidf_matrix).argmax(axis=1)

# Evaluate clustering performance
true_labels = np.array(labels)

# K-Means evaluation
kmeans_ari = adjusted_rand_score(true_labels, kmeans_labels)
kmeans_nmi = normalized_mutual_info_score(true_labels, kmeans_labels)

# LDA evaluation
lda_ari = adjusted_rand_score(true_labels, lda_labels)
lda_nmi = normalized_mutual_info_score(true_labels, lda_labels)

print("\n" + "="*60)
print("CLUSTERING EVALUATION RESULTS")
print("="*60)
print(f"K-Means - Adjusted Rand Index: {kmeans_ari:.3f}")
print(f"K-Means - Normalized Mutual Information: {kmeans_nmi:.3f}")
print(f"LDA - Adjusted Rand Index: {lda_ari:.3f}")
print(f"LDA - Normalized Mutual Information: {lda_nmi:.3f}")

# Display top words for each topic
def print_top_words(model, feature_names, n_top_words=10):
    for topic_idx, topic in enumerate(model.components_):
        top_features_ind = topic.argsort()[-n_top_words:][::-1]
        top_features = [feature_names[i] for i in top_features_ind]
        weights = [topic[i] for i in top_features_ind]
        print(f"Topic {topic_idx}: {', '.join(top_features)}")

print("\n" + "="*60)
print("LDA TOPICS - TOP WORDS")
print("="*60)
print_top_words(lda, feature_names, n_top_words=8)

# For K-Means, get top words by cluster centroids
print("\n" + "="*60)
print("K-MEANS CLUSTERS - TOP WORDS")
print("="*60)
for cluster_idx in range(20):
    centroid = kmeans.cluster_centers_[cluster_idx]
    top_features_ind = centroid.argsort()[-8:][::-1]
    top_features = [feature_names[i] for i in top_features_ind]
    print(f"Cluster {cluster_idx}: {', '.join(top_features)}")

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
import numpy as np
from collections import Counter
import matplotlib.pyplot as plt
from topicgpt.ExtractTopWords import ExtractTopWords
from topicgpt.GetEmbeddingsOpenAI import GetEmbeddingsOpenAI

# Load your dataset
from sklearn.datasets import fetch_20newsgroups

data = fetch_20newsgroups(subset='all', remove=('headers', 'footers', 'quotes'))
corpus_data = data['data']
topic_names = data.target_names

filtered_corpus = []
filtered_labels = []

for doc, label in zip(corpus_data, data.target):
    if doc != "":
        filtered_corpus.append(doc)
        filtered_labels.append(label)

corpus = filtered_corpus
labels = filtered_labels

print("Preparing data for topic modeling...")
print(f"Number of documents: {len(corpus)}")
print(f"Number of true topics: {len(topic_names)}")

# Create TF-IDF features
print("\nCreating TF-IDF features...")
vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words='english',
    min_df=2,
    max_df=0.95,
    ngram_range=(1, 2)
)
tfidf_matrix = vectorizer.fit_transform(corpus)
feature_names = vectorizer.get_feature_names_out()
print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")

# K-Means Clustering
print("\nPerforming K-Means clustering...")
kmeans = KMeans(n_clusters=20, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(tfidf_matrix)

# LDA Topic Modeling
print("Performing LDA topic modeling...")
lda = LatentDirichletAllocation(
    n_components=20,
    random_state=42,
    max_iter=100,
    learning_method='batch'
)
lda.fit(tfidf_matrix)
lda_labels = lda.transform(tfidf_matrix).argmax(axis=1)

# Initialize ExtractTopWords for custom scoring
extractor = ExtractTopWords()

# Compute vocabulary using ExtractTopWords methods
print("\nComputing vocabulary...")
vocab = extractor.compute_corpus_vocab(
    corpus,
    remove_stopwords=True,
    remove_punction=True,
    min_word_length=3,
    max_word_length=20,
    remove_numbers=True,
    min_doc_frequency=3,
    min_freq=0.1,
    max_freq=0.9,
    verbose=True
)

print(f"Vocabulary size: {len(vocab)}")

# Get embeddings for vocabulary (you'll need OpenAI API key)
# Uncomment and add your API key if you want cosine similarity scores
"""
from topicgpt.clients import OpenAIClient
client = OpenAIClient(api_key="your_api_key_here")
embedder = GetEmbeddingsOpenAI(client)
vocab_embeddings = extractor.embed_vocab(vocab, embedder)
"""

# Compute word-topic matrices for both methods
print("\nComputing word-topic matrices...")

# For K-Means
kmeans_word_topic_mat = extractor.compute_word_topic_mat(
    corpus, vocab, kmeans_labels, consider_outliers=False
)

# For LDA
lda_word_topic_mat = extractor.compute_word_topic_mat(
    corpus, vocab, lda_labels, consider_outliers=False
)

# Extract top words with TF-IDF scores
print("\nExtracting top words with TF-IDF scores...")

# K-Means TF-IDF scores
kmeans_top_words_tfidf, kmeans_top_scores_tfidf = extractor.extract_topwords_tfidf(
    kmeans_word_topic_mat, vocab, kmeans_labels, top_n_words=15
)

# LDA TF-IDF scores
lda_top_words_tfidf, lda_top_scores_tfidf = extractor.extract_topwords_tfidf(
    lda_word_topic_mat, vocab, lda_labels, top_n_words=15
)

# Evaluation
true_labels = np.array(labels)

kmeans_ari = adjusted_rand_score(true_labels, kmeans_labels)
kmeans_nmi = normalized_mutual_info_score(true_labels, kmeans_labels)

lda_ari = adjusted_rand_score(true_labels, lda_labels)
lda_nmi = normalized_mutual_info_score(true_labels, lda_labels)

print("\n" + "="*80)
print("CLUSTERING EVALUATION RESULTS")
print("="*80)
print(f"K-Means - Adjusted Rand Index: {kmeans_ari:.3f}")
print(f"K-Means - Normalized Mutual Information: {kmeans_nmi:.3f}")
print(f"LDA - Adjusted Rand Index: {lda_ari:.3f}")
print(f"LDA - Normalized Mutual Information: {lda_nmi:.3f}")

# Display results with detailed scores
print("\n" + "="*80)
print("K-MEANS CLUSTERING - TOP WORDS WITH TF-IDF SCORES")
print("="*80)

for topic_id in sorted(kmeans_top_words_tfidf.keys()):
    print(f"\nCluster {topic_id}:")
    print("-" * 40)
    words = kmeans_top_words_tfidf[topic_id][:10]
    scores = kmeans_top_scores_tfidf[topic_id][:10]
    
    for word, score in zip(words, scores):
        print(f"  {word:15s} | TF-IDF: {score:.4f}")

print("\n" + "="*80)
print("LDA TOPIC MODELING - TOP WORDS WITH TF-IDF SCORES")
print("="*80)

for topic_id in sorted(lda_top_words_tfidf.keys()):
    print(f"\nTopic {topic_id}:")
    print("-" * 40)
    words = lda_top_words_tfidf[topic_id][:10]
    scores = lda_top_scores_tfidf[topic_id][:10]
    
    for word, score in zip(words, scores):
        print(f"  {word:15s} | TF-IDF: {score:.4f}")

# Compare with standard LDA topic words
print("\n" + "="*80)
print("STANDARD LDA TOPICS (for comparison)")
print("="*80)

def print_top_words_standard(model, feature_names, n_top_words=10):
    for topic_idx, topic in enumerate(model.components_):
        top_features_ind = topic.argsort()[-n_top_words:][::-1]
        top_features = [feature_names[i] for i in top_features_ind]
        weights = [topic[i] for i in top_features_ind]
        print(f"\nTopic {topic_idx}:")
        print("-" * 40)
        for word, weight in zip(top_features, weights):
            print(f"  {word:15s} | Weight: {weight:.4f}")

print_top_words_standard(lda, feature_names, n_top_words=10)

# Optional: If you have embeddings, compute cosine similarity scores
"""
if 'vocab_embeddings' in locals():
    print("\n" + "="*80)
    print("COSINE SIMILARITY SCORES (requires embeddings)")
    print("="*80)
    
    # Extract centroids for K-Means (you'll need document embeddings for this)
    # This requires computing document embeddings first
    
    # For demonstration, showing how it would work:
    # kmeans_centroids = extractor.extract_centroids(document_embeddings, kmeans_labels)
    # kmeans_top_words_cosine, kmeans_top_scores_cosine = extractor.extract_topwords_centroid_similarity(
    #     kmeans_word_topic_mat, vocab, vocab_embeddings, kmeans_centroids, 
    #     umap_mapper, top_n_words=15
    # )
"""

# Summary statistics
print("\n" + "="*80)
print("SUMMARY STATISTICS")
print("="*80)
print(f"Vocabulary size after filtering: {len(vocab)}")
print(f"K-Means - Number of clusters found: {len(np.unique(kmeans_labels))}")
print(f"LDA - Number of topics: {lda.n_components}")
print(f"Average words per K-Means cluster: {np.mean([len(words) for words in kmeans_top_words_tfidf.values()]):.1f}")
print(f"Average words per LDA topic: {np.mean([len(words) for words in lda_top_words_tfidf.values()]):.1f}")

# Function to get top words for a specific cluster/topic
def get_topic_summary(method, topic_id, n_words=5):
    """Get a summary of top words for a specific topic."""
    if method.lower() == 'kmeans':
        words = kmeans_top_words_tfidf.get(topic_id, [])[:n_words]
        scores = kmeans_top_scores_tfidf.get(topic_id, [])[:n_words]
    elif method.lower() == 'lda':
        words = lda_top_words_tfidf.get(topic_id, [])[:n_words]
        scores = lda_top_scores_tfidf.get(topic_id, [])[:n_words]
    else:
        return "Invalid method. Use 'kmeans' or 'lda'."
    
    return list(zip(words, scores))

# Example usage
print(f"\nExample - K-Means Cluster 0 top 5 words:")
print(get_topic_summary('kmeans', 0, 5))

print(f"\nExample - LDA Topic 0 top 5 words:")
print(get_topic_summary('lda', 0, 5))

## Using BertTopic

In [5]:
def chunk_text(text, tokenizer, max_tokens):
    """Split text into chunks, each <= max_tokens (by tokens, not words)."""
    tokens = tokenizer.encode(text)
    chunks = []
    for i in range(0, len(tokens), max_tokens):
        chunk_tokens = tokens[i:i+max_tokens]
        chunk_text = tokenizer.decode(chunk_tokens)
        chunks.append(chunk_text)
    return chunks

In [ ]:
from bertopic import BERTopic
import tiktoken

## constants
MAX_TOKENS = 1024  # max tokens per chunk
tokenizer = tiktoken.encoding_for_model("text-embedding-ada-002")
new_corpus = []
for docs in corpus:
    chunks = chunk_text(docs, tokenizer, MAX_TOKENS)
    new_corpus.extend(chunks)

# Pre-calculate embeddings
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedding_model.encode(new_corpus, show_progress_bar=True)

# preventing stochastic behaviour for reproducibility
from umap import UMAP
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42)

# controlling the number of topics
from hdbscan import HDBSCAN
hdbscan_model = HDBSCAN(min_cluster_size=int(len(corpus)/150), metric='euclidean', cluster_selection_method='eom', prediction_data=True)

# improve default representation
from sklearn.feature_extraction.text import CountVectorizer
vectorizer_model = CountVectorizer(stop_words="english", min_df=2, ngram_range=(1, 2))

# additional representations
import openai
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance, OpenAI, PartOfSpeech
keybert_model = KeyBERTInspired() # KeyBERT
# pos_model = PartOfSpeech("en_core_web_sm") # Part-of-Speech
mmr_model = MaximalMarginalRelevance(diversity=0.3) # MMR
# GPT-3.5
prompt = """
I have a topic that contains the following documents:
[DOCUMENTS]
The topic is described by the following keywords: [KEYWORDS]

Based on the information above, extract a short but highly descriptive topic label of at most 5 words. Make sure it is in the following format:
topic: <topic label>
"""

client = openai.OpenAI(api_key=api_key)
openai_model = OpenAI(
                client, 
                model="gpt-3.5-turbo", 
                exponential_backoff=True, 
                chat=True, 
                prompt=prompt,
                nr_repr_docs=3 # Only use 3 representative docs per topic
            )

# All representation models
representation_model = {
    "KeyBERT": keybert_model,
    "OpenAI": openai_model,  # Uncomment if you will use OpenAI
    "MMR": mmr_model,
    # "POS": pos_model
}

### TRAINING
from bertopic import BERTopic

topic_model = BERTopic(
        # Pipeline models
        embedding_model=embedding_model,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        representation_model=representation_model,
        # Hyperparameters
        top_n_words=10,
        nr_topics=20,
        verbose=True
)

topics, probs = topic_model.fit_transform(new_corpus, embeddings)

Batches:   0%|          | 0/658 [00:00<?, ?it/s]

2025-11-12 09:28:18,259 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-11-12 09:28:37,709 - BERTopic - Dimensionality - Completed ✓
2025-11-12 09:28:37,711 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-11-12 09:28:39,224 - BERTopic - Cluster - Completed ✓
2025-11-12 09:28:39,226 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2025-11-12 09:28:42,085 - BERTopic - Representation - Completed ✓
2025-11-12 09:28:42,088 - BERTopic - Topic reduction - Reducing number of topics
2025-11-12 09:28:42,088 - BERTopic - Topic reduction - Number of topics (20) is equal or higher than the clustered topics(17).
2025-11-12 09:28:42,088 - BERTopic - Representation - Fine-tuning topics using representation models.
100%|██████████| 17/17 [00:19<00:00,  1.12s/it]
2025-11-12 09:29:06,943 - BERTopic - Representation - Completed ✓


In [7]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,KeyBERT,OpenAI,MMR,Representative_Docs
0,-1,2183,-1_ax_ax ax_people_db,"[ax, ax ax, people, db, don, think, like, just...","[morality, objective, moral, anonymous, islam,...",[Objective Morality Debate],"[ax, ax ax, think, god, government, moral, inf...",[#>In article <1993Apr20.070156.26910@abo.fi> ...
1,0,150,0_hello_hi_ites_cheek,"[hello, hi, ites, cheek, yep, huh, ic, ken, go...","[hello, hi, , , , , , , , ]",[Greetings],"[hello, hi, ites, cheek, yep, huh, ic, ken, go...","[Hi,, Hello,, Hello,]"
2,1,2043,1_game_team_games_year,"[game, team, games, year, season, hockey, play...","[nhl, hockey, playoffs, puck, montreal, flyers...",[NHL Playoff Schedule],"[games, season, hockey, players, league, 12, p...",[ ESPN to televise up to 25 \nregular-season g...
3,2,294,2_ax_ax ax_max_max ax,"[ax, ax ax, max, max ax, ax max, g9v, b8f, g9v...","[max ax, ax max, ax ax, ax, ax qq, qax ax, bhj...",[Max Variations and Patterns],"[ax ax, max, max ax, ax max, g9v g9v, a86, pl,...",[AX>'AX>'AX>'AX>'AX>'AX>'AX>'\nMAX>'AX>'AX>'AX...
4,3,474,3_ax_ax ax_cx_34u,"[ax, ax ax, cx, 34u, ah, mv, c_, w7, chz, ck]","[max ax, ax max, lhz, mz, chz, m3, m4, m0, m2,...",[Encrypted Data Analysis],"[cx, 34u, w7, mw, hz, mq, ax max, a86, 1d9, 7u]","[-FMK""<_/S\_/SX#/S\_/S\_/SU=76EJ]O5JB6W5U\nMGG..."
5,4,983,4_health_medical_cancer_patients,"[health, medical, cancer, patients, disease, h...","[hiv, aids, diseases, health, medical newslett...",[Medical Resources and Information],"[health, medical, patients, hiv, vitamin, msg,...",[fat and very few animal and dairy products. ...
6,5,6661,5_windows_use_file_dos,"[windows, use, file, dos, edu, software, drive...","[mac, hardware, graphics, ram, os, software, j...",[Windows Software Solutions],"[windows, dos, software, files, graphics, vers...","[Pixie, called\nPICTCompressor, floating aroun..."
7,6,1880,6_car_bike_just_cars,"[car, bike, just, cars, engine, like, don, goo...","[vehicle, driving, steering, riding, brake, pa...",[Car and motorcycle experiences],"[bike, cars, engine, road, riding, rear, deale...",[...\n...\n\nSome other owners on the ford-pro...
8,7,697,7_israel_israeli_jews_arab,"[israel, israeli, jews, arab, jewish, people, ...","[palestinians, israeli, israelis, palestinian,...",[Israeli-Palestinian Conflict and Identity],"[israel, israeli, peace, palestinian, palestin...",[From: Center for Policy Research <cpr>\nSubje...
9,8,429,8_armenian_armenians_turkish_people,"[armenian, armenians, turkish, people, said, t...","[armenians, armenian, armenian government, arm...",[Armenian-Turkish genocide dispute],"[armenian, armenians, turkish, turkey, armenia...",[\n\nThey are news because they are the except...


In [8]:
topic_model.get_topic(1, full=True)

{'Main': [('game', np.float64(0.031179815867109748)),
  ('team', np.float64(0.026315457956133832)),
  ('games', np.float64(0.01948876162745426)),
  ('year', np.float64(0.018687886642993534)),
  ('season', np.float64(0.01697417320076128)),
  ('hockey', np.float64(0.016435613446241042)),
  ('play', np.float64(0.016187932881595632)),
  ('25', np.float64(0.016097219504443684)),
  ('10', np.float64(0.01573670444395984)),
  ('players', np.float64(0.015729950187464054))],
 'KeyBERT': [('nhl', np.float32(0.6127897)),
  ('hockey', np.float32(0.51132494)),
  ('playoffs', np.float32(0.48731917)),
  ('puck', np.float32(0.4806475)),
  ('montreal', np.float32(0.45684874)),
  ('flyers', np.float32(0.439981)),
  ('rangers', np.float32(0.39329857)),
  ('espn', np.float32(0.34813875)),
  ('toronto', np.float32(0.32761723)),
  ('teams', np.float32(0.30439204))],
 'OpenAI': [('NHL Playoff Schedule', 1)],
 'MMR': [('games', np.float64(0.01948876162745426)),
  ('season', np.float64(0.01697417320076128)),
  

In [9]:
freq = topic_model.get_topic_info(); freq.head(5)

,Topic,Count,Name,Representation,KeyBERT,OpenAI,MMR,Representative_Docs
0,-1,2183,-1_ax_ax ax_people_db,"[ax, ax ax, people, db, don, think, like, just...","[morality, objective, moral, anonymous, islam,...",[Objective Morality Debate],"[ax, ax ax, think, god, government, moral, inf...",[#>In article <1993Apr20.070156.26910@abo.fi> ...
1,0,150,0_hello_hi_ites_cheek,"[hello, hi, ites, cheek, yep, huh, ic, ken, go...","[hello, hi, , , , , , , , ]",[Greetings],"[hello, hi, ites, cheek, yep, huh, ic, ken, go...","[Hi,, Hello,, Hello,]"
2,1,2043,1_game_team_games_year,"[game, team, games, year, season, hockey, play...","[nhl, hockey, playoffs, puck, montreal, flyers...",[NHL Playoff Schedule],"[games, season, hockey, players, league, 12, p...",[ ESPN to televise up to 25 \nregular-season g...
3,2,294,2_ax_ax ax_max_max ax,"[ax, ax ax, max, max ax, ax max, g9v, b8f, g9v...","[max ax, ax max, ax ax, ax, ax qq, qax ax, bhj...",[Max Variations and Patterns],"[ax ax, max, max ax, ax max, g9v g9v, a86, pl,...",[AX>'AX>'AX>'AX>'AX>'AX>'AX>'\nMAX>'AX>'AX>'AX...
4,3,474,3_ax_ax ax_cx_34u,"[ax, ax ax, cx, 34u, ah, mv, c_, w7, chz, ck]","[max ax, ax max, lhz, mz, chz, m3, m4, m0, m2,...",[Encrypted Data Analysis],"[cx, 34u, w7, mw, hz, mq, ax max, a86, 1d9, 7u]","[-FMK""<_/S\_/SX#/S\_/S\_/SU=76EJ]O5JB6W5U\nMGG..."


## Using TopicGPT

In [ ]:
from topicgpt.TopicGPT import TopicGPT
if TRAIN:
    tm = TopicGPT(
        prompting_model=prompting_model,
        api_key=api_key,
        n_topics=20,  # select 20 topics since the true number of topics is 20
        embedding_model=embedding_model, 
        use_saved_embeddings=False,  # set to False to train the model from scratch
    )
    tm.fit(corpus)  # train the model on the corpus
    tm.save_embeddings(EMBEDDING_PATH) #save the embeddings for future use
else:
    tm = TopicGPT(
        prompting_model=prompting_model,
        embedding_model=embedding_model, 
        api_key=api_key,
        n_topics=20,  # select 20 topics since the true number of topics is 20
        path_saved_embeddings=EMBEDDING_PATH,
        use_saved_embeddings=True,  # set to True to use saved embeddings
    )

In [ ]:
tm

In [ ]:
tm.extract_topics(corpus)

In [ ]:
tm.describe_topics(tm.topic_lis)

In [ ]:
# index = 1
# print(tm.topic_lis[index].topic_name)
# print(tm.topic_lis[index].topic_description)


# import regex as re
# def clean_description(desc: str) -> str:
#     if not desc:
#         return ""
#     # remove surrounding quotes
#     desc = desc.strip().strip('\'"')
#     # remove markdown bold/italic and inline code
#     desc = re.sub(r'(\*\*|\*|`)+', '', desc)
#     # remove headings like "##" or "**Aspects**:"
#     desc = re.sub(r'^#{1,6}\s*', '', desc, flags=re.MULTILINE)
#     # remove numbered list numbering (1. , 2. ), bullets (-, *, +)
#     desc = re.sub(r'^\s*[\-\*\+]\s+', '', desc, flags=re.MULTILINE)
#     desc = re.sub(r'^\s*\d+\.\s+', '', desc, flags=re.MULTILINE)
#     # replace multiple newlines/whitespace with single space
#     desc = re.sub(r'\s+', ' ', desc)
#     return desc.strip()

# cleaned_description = clean_description(tm.topic_lis[index].topic_description)
# cleaned_description

In [ ]:
# import numpy as np
# print(np.array(filtered_labels).shape)
# print(topic_names)
# import numpy as np
# dim_red_embeddings, labels, umap_mapper = tm.clusterer.cluster_and_reduce(tm.document_embeddings)  # get dimensionality reduced embeddings, their labels and the umap mapper object

# unique_labels = np.unique(labels)  # In case the cluster labels are not consecutive numbers, we need to map them to consecutive 
# label_mapping = {label: i for i, label in enumerate(unique_labels[unique_labels != -1])}
# label_mapping[-1] = -1
# pred_labels = np.array([label_mapping[label] for label in labels])

In [ ]:
# import numpy as np
# print(np.array(filtered_labels).shape)
# print(topic_names)
# import numpy as np
# dim_red_embeddings, labels, umap_mapper = tm.clusterer.cluster_and_reduce(tm.document_embeddings)  # get dimensionality reduced embeddings, their labels and the umap mapper object

# unique_labels = np.unique(labels)  # In case the cluster labels are not consecutive numbers, we need to map them to consecutive 
# label_mapping = {label: i for i, label in enumerate(unique_labels[unique_labels != -1])}
# label_mapping[-1] = -1
# labels = np.array([label_mapping[label] for label in labels])
# predicted_labels = labels
# print(np.unique(np.array(predicted_labels)))  # The predicted labels are between 0 and 19
# print(predicted_labels.shape)
# true_labels = np.array(filtered_labels)

In [ ]:
# from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

# ari = adjusted_rand_score(true_labels, predicted_labels)
# nmi = normalized_mutual_info_score(true_labels, predicted_labels)

# print(f"Adjusted Rand Index: {ari:.3f}")
# print(f"Normalized Mutual Information: {nmi:.3f}")

In [ ]:
tm.score(100, 200)

In [ ]:
# Test ADC metric with dummy data and print shapes at key steps

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Dummy Topic class with required fields
class DummyTopic:
    def __init__(self, doc_embs, centroid=None, docs=None):
        self.document_embeddings_hd = doc_embs
        self.centroid_hd = centroid
        self.documents = docs if docs is not None else ["doc" + str(i) for i in range(len(doc_embs))]

# Create dummy topics
np.random.seed(42)
topic1_embs = np.random.rand(3, 5)  # 3 docs, 5-dim
topic2_embs = np.random.rand(2, 5)  # 2 docs, 5-dim
topic3_embs = np.random.rand(4, 5)  # 4 docs, 5-dim

topic1 = DummyTopic(topic1_embs)
topic2 = DummyTopic(topic2_embs)
topic3 = DummyTopic(topic3_embs)

topic_list = [topic1, topic2, topic3]

# Import ADC metric
from topicgpt.metrics.intruder_metrics import ADC

adc = ADC(n_docs=-1, n_intruder_docs=2)

# --- Insert print statements in ADC methods for debugging ---
import types

import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

def debug_score_one_intr_per_cluster(self, topic_list, new_embeddings=True, random_state=None):
    rng = np.random.default_rng(random_state)
    emb_clusters = []
    for t in topic_list:
        emb = getattr(t, "document_embeddings_hd", None)
        arr = np.atleast_2d(np.asarray(emb))
        emb_clusters.append(arr)
    scores = []
    for i, cluster_emb in enumerate(emb_clusters):
        print(f"\nProcessing cluster {i}, cluster_emb shape: {cluster_emb.shape}")
        if cluster_emb.size == 0:
            scores.append(np.nan)
            continue
        other = [np.atleast_2d(c) for j, c in enumerate(emb_clusters) if j != i and c.size > 0]
        if len(other) == 0:
            scores.append(np.nan)
            continue
        other_embs = np.vstack(other)
        intr_idx = int(rng.integers(0, other_embs.shape[0]))
        intr_embedding = other_embs[intr_idx]
        print(f"Intruder doc index in other_embs: {intr_idx}")
        # Visualize
        pca = PCA(n_components=2)
        all_points = np.vstack([cluster_emb, intr_embedding.reshape(1, -1)])
        points_2d = pca.fit_transform(all_points)
        plt.figure(figsize=(5, 5))
        plt.scatter(points_2d[:-1, 0], points_2d[:-1, 1], c='blue', label='Cluster Docs')
        plt.scatter(points_2d[-1, 0], points_2d[-1, 1], c='red', marker='*', s=200, label='Intruder')
        plt.title(f'Cluster {i} vs Intruder')
        plt.legend()
        plt.show()
        # Continue with similarity calculation
        sim = cosine_similarity(intr_embedding.reshape(1, -1), cluster_emb)
        scores.append(float(np.mean(sim)))
    return np.array(scores)

# Patch the method for debugging
adc.score_one_intr_per_cluster = types.MethodType(debug_score_one_intr_per_cluster, adc)

# Run the metric and observe the printouts
print("=== Running ADC metric on dummy data ===")
result = adc.score(topic_list)
print("\nFinal ADC score:", result)

In [34]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import warnings

class Topic:
    def __init__(self, document_embeddings_hd, centroid_hd=None, documents=None):
        self.document_embeddings_hd = document_embeddings_hd
        self.centroid_hd = centroid_hd
        self.documents = documents or []

class ADC:
    """
    Average Document Coherence (ADC) metric for topic models.
    """

    def __init__(
        self,
        n_docs=-1,  # -1 means all docs in the cluster
        n_intruder_docs=1,
    ):
        self.n_docs = n_docs
        self.n_intruder_docs = n_intruder_docs
        # self.score_one_intr_per_cluster_count = 0 ### debug print

        info = {
            "metric_name": "Average Document Coherence (ADC) ",
            "n_docs": self.n_docs,
            "n_intruder_docs": self.n_intruder_docs,
            "metric_range": "0 to 1, smaller is better",
            "description": " the average cosine similarity between every word in a topic and an intruder word.",
        }
        return info

    def score_one_intr_per_cluster(
        self,
        topic_list,
        random_state=None,
    ):
        # self.score_one_intr_per_cluster_count += 1 ### debug print
        print(f"score_one_intr_per_cluster called {self.score_one_intr_per_cluster_count} times") ### debug print

        rng = np.random.default_rng(random_state)
        emb_clusters = []
        for t in topic_list:
            emb = getattr(t, "document_embeddings_hd", None)
            assert emb is not None, "Topic objects must have document_embeddings_hd field populated."
            arr = np.atleast_2d(np.asarray(emb))
            if self.n_docs is not None and self.n_docs > 0 and arr.size > 0:
                n_available = arr.shape[0]
                if n_available > self.n_docs:
                    centroid = getattr(t, "centroid_hd", None)
                    if centroid is None:
                        centroid = arr.mean(axis=0)
                    else:
                        centroid = np.asarray(centroid).reshape(-1)
                    sims = cosine_similarity(centroid.reshape(1, -1), arr).flatten()
                    top_idx = np.argsort(sims)[-self.n_docs:][::-1]
                    arr = arr[top_idx]
            emb_clusters.append(arr)

        scores = []
        for i, cluster_emb in enumerate(emb_clusters):
            # print(f'CLUSTER {i}\n----------------------') ### debug print
            if cluster_emb.size == 0:
                scores.append(np.nan)
                continue
            # print(f'cluster_{i} shape: {cluster_emb.shape}') ### debug print
            other = [np.atleast_2d(c) for j, c in enumerate(emb_clusters) if j != i and c.size > 0]
            if len(other) == 0:
                scores.append(np.nan)
                continue
            other_embs = np.vstack(other)
            # print(f'other_embs_{i} shape: {other_embs.shape}') ### debug print
            intr_idx = int(rng.integers(0, other_embs.shape[0]))
            intr_embedding = other_embs[intr_idx]
            # compute similarity between intruder and all docs in current topic
            sim = cosine_similarity(intr_embedding.reshape(1, -1), cluster_emb)  # (1, n_docs)
            sim_scaled = (sim + 1) / 2  # Scale to [0, 1]
            scores.append(float(np.mean(sim_scaled)))
        return np.array(scores)

    def score_per_cluster(self, topic_list):
        score_lis = []
        for _ in range(self.n_intruder_docs):
            score_per_cluster = self.score_one_intr_per_cluster(
                topic_list
            )
            score_lis.append(score_per_cluster)
        res = np.vstack(score_lis).T
        mean_scores = np.mean(res, axis=1)
        ntopics = len(topic_list)
        results = {}
        for k in range(ntopics):
            preview = ""
            t = topic_list[k]
            if getattr(t, "documents", None):
                preview = " - " + str(t.documents[0])[:40].replace("\n", " ").strip()
            label = f"cluster_{k}{preview}"
            results[label] = float(np.round(mean_scores[k], 5))
        return results

    def score(self, topics):
        scores = list(self.score_per_cluster(topics).values())
        if all(np.isnan(scores)):
            warnings.warn("ADC: All clusters returned NaN (no valid intruder comparisons possible).")
            return np.nan
        return float(np.nanmean(scores))

In [33]:
# Dummy input for extreme cases

# Case 1: All clusters empty
topics_empty = [Topic(document_embeddings_hd=np.array([])) for _ in range(3)]

# Case 2: One cluster with docs, others empty
topics_one = [
    Topic(document_embeddings_hd=np.random.randn(5, 10), documents=["doc1"]*5),
    Topic(document_embeddings_hd=np.array([])),
    Topic(document_embeddings_hd=np.array([])),
]

# Case 3: All clusters with one doc each
topics_single = [
    Topic(document_embeddings_hd=np.random.randn(1, 10), documents=["docA"]),
    Topic(document_embeddings_hd=np.random.randn(1, 10), documents=["docB"]),
    Topic(document_embeddings_hd=np.random.randn(1, 10), documents=["docC"]),
]

# Case 4: Normal case, multiple docs per cluster
topics_normal = [
    Topic(document_embeddings_hd=np.random.randn(5, 10), documents=["doc1"]*5),
    Topic(document_embeddings_hd=np.random.randn(6, 10), documents=["doc2"]*6),
    Topic(document_embeddings_hd=np.random.randn(4, 10), documents=["doc3"]*4),
]

adc = ADC(n_docs=-1, n_intruder_docs=2)

# print("All clusters empty:", adc.score(topics_empty))
# print("One cluster with docs, others empty:", adc.score(topics_one))
# print("All clusters with one doc each:", adc.score(topics_single))
print("Normal case:", adc.score(topics_normal))

# You can also inspect topics_one if you want:
print("\ntopics_one:")
for idx, t in enumerate(topics_one):
    print(f"Cluster {idx}: embeddings shape {np.array(t.document_embeddings_hd).shape}")

score_one_intr_per_cluster called 1 times
CLUSTER 0
----------------------
embedding_cluster_0 shape: (5, 10)
other_embs_0 shape: (10, 10)
CLUSTER 1
----------------------
embedding_cluster_1 shape: (6, 10)
other_embs_1 shape: (9, 10)
CLUSTER 2
----------------------
embedding_cluster_2 shape: (4, 10)
other_embs_2 shape: (11, 10)
score_one_intr_per_cluster called 2 times
CLUSTER 0
----------------------
embedding_cluster_0 shape: (5, 10)
other_embs_0 shape: (10, 10)
CLUSTER 1
----------------------
embedding_cluster_1 shape: (6, 10)
other_embs_1 shape: (9, 10)
CLUSTER 2
----------------------
embedding_cluster_2 shape: (4, 10)
other_embs_2 shape: (11, 10)
Normal case: 0.5098833333333334

topics_one:
Cluster 0: embeddings shape (5, 10)
Cluster 1: embeddings shape (0,)
Cluster 2: embeddings shape (0,)


In [ ]:
### STANDALONE ADS METRIC 
import numpy as np
import re
from sklearn.metrics.pairwise import cosine_similarity

class ADS:
    """
    Average Description Similarity (ADS) metric for topic models.
    The average cosine similarity of the embedding of each document d
    to the embedding of the description of the topic assigned to d.
    """

    def __init__(self, n_docs: int = -1, embedder=None):
        """
        Args:
            n_docs (int): Number of documents per topic to use (-1 means all).
            embedder: An object with a .get_embeddings(list_of_texts) method returning a dict with 'embeddings' key.
        """
        self.n_docs = n_docs
        self.embedder = embedder

    def get_info(self):
        """
        Get information about the metric.
        """
        info = {
            "metric_name": "Average Description Similarity (ADS)",
            "n_docs": self.n_docs,
            "metric_range": "0 to 1, higher is better",
            "description": "The average cosine similarity (scaled to [0,1]) between the embedding of each document and the embedding of its topic description.",
        }
        return info

    def score(self, topics):
        """
        Args:
            topics: List of Topic-like objects, each with:
                - topic_description (str)
                - document_embeddings_hd (np.ndarray)
                - centroid_hd (optional, np.ndarray)
        Returns:
            float: The average ADS score across all topics.
        """
        assert isinstance(topics, (list, tuple)), "topics must be a list or tuple of Topic objects"
        assert self.embedder is not None, "embedder must be provided"

        # Clean the descriptions
        descriptions = [self.clean_description(getattr(t, "topic_description", "")) for t in topics]

        # Embed the topic descriptions
        topic_desc_embeddings = self.embedder.get_embeddings(descriptions)["embeddings"]

        # Embed the documents in each topic
        emb_clusters = []
        for t in topics:
            emb = getattr(t, "document_embeddings_hd", None)
            assert emb is not None, "Each topic must have 'document_embeddings_hd'"
            arr = np.atleast_2d(np.asarray(emb))
            # If n_docs > 0, select the most representative documents (top-k by similarity to centroid)
            if self.n_docs is not None and self.n_docs > 0 and arr.size > 0:
                n_available = arr.shape[0]
                n_select = min(self.n_docs, n_available)
                centroid = getattr(t, "centroid_hd", None)
                if centroid is None:
                    centroid = arr.mean(axis=0)
                else:
                    centroid = np.asarray(centroid).reshape(-1)
                sims = cosine_similarity(centroid.reshape(1, -1), arr).flatten()
                top_idx = np.argsort(sims)[-n_select:][::-1]
                arr = arr[top_idx]
            emb_clusters.append(arr)

        similarity_scores = []
        for i in range(len(emb_clusters)):
            desc = topic_desc_embeddings[i].reshape(1, -1)
            docs = emb_clusters[i]
            if docs.size == 0:
                similarity_scores.append(np.nan)
                continue
            sims = cosine_similarity(desc, docs)  # Shape: (1, n_docs)
            sims_01 = (sims + 1) / 2  # Now in [0, 1]
            similarity_scores.append(np.nanmean(sims_01))  # Average similarity for the topic

        if len(similarity_scores) == 0 or np.all(np.isnan(similarity_scores)):
            return np.nan
        return round(np.nanmean(similarity_scores), 4)

    @staticmethod
    def clean_description(desc: str) -> str:
        if not desc:
            return ""
        desc = desc.strip().strip('\'"')
        desc = re.sub(r'(\*\*|\*|`)+', '', desc)
        desc = re.sub(r'^#{1,6}\s*', '', desc, flags=re.MULTILINE)
        desc = re.sub(r'^\s*[\-\*\+]\s+', '', desc, flags=re.MULTILINE)
        desc = re.sub(r'^\s*\d+\.\s+', '', desc, flags=re.MULTILINE)
        desc = re.sub(r'\s+', ' ', desc)
        return desc.strip()